In [1]:
# GPU 사용 가능 여부 확인
import sys
import torch
import random
import numpy as np
from ultralytics import YOLO

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Python version: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
PyTorch version: 2.7.1+cu118
CUDA version: 11.8
GPU available: True
GPU name: NVIDIA GeForce RTX 4090


In [2]:
# 모델 재현을 위한 랜덤시드 고정
def set_random_seed(seed_value=42):
    # Python의 기본 난수 시드 설정
    random.seed(seed_value)
    # NumPy 난수 시드 설정
    np.random.seed(seed_value)
    # PyTorch 난수 시드 설정 (CPU)
    torch.manual_seed(seed_value)
    # PyTorch 난수 시드 설정 (GPU)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    # CuDNN 설정
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed()

# matplotlib default 설정
import matplotlib.pyplot as plt
plt.style.use('default')

In [3]:
# 1. 모델 불러오기
model = YOLO('yolo11m.pt')

# train 파라미터 설정
train_params = {
    'data': "/home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/data.yaml",
    'epochs': 150,
    'imgsz': 640,
    'batch': 16,
    'project': './runs',
    'name': 'yolov11m',
    'seed': 42,
    'optimizer': 'AdamW',           # 동일하게 AdamW 사용
    'weight_decay': 1e-4,
    'cos_lr': True,
    'warmup_epochs': int(0.05*150), # hf 설정과 유사하게 전체 epoch의 5%를 warmup에 사용
    'lr0': 0.001,                   # 공식 권장값이 1e-3
    'lrf': 0.01,
    'workers': 12,                   # default: 8
    'patience': 40,

    # [데이터 증강 (HF와 공정한 비교를 위해 통제)]
    'hsv_h': 0.015,  # 참외 노란색 보존
    'hsv_s': 0.5,
    'hsv_v': 0.4,
    'degrees': 10.0,
    'translate': 0.1,
    'scale': 0.9,    # HF의 crop/affine 줌인/아웃과 유사한 강도
    'fliplr': 0.5,
    'mosaic': 0.0,   # 공정성을 위해 YOLO 필살기 끄기
    'mixup': 0.0,
    'erasing': 0.2
}

In [4]:
# 모델 학습
results = model.train(**train_params)

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24091MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov11m, nbs=64, nms=False, opset=None, optimi

/home/leedh/bin/miniconda3/envs/paper_exp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4069.5±1427.2 MB/s, size: 6956.6 KB)
val: Scanning /home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/labels/val... 154 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 154/154 865.1it/s 0.2s.1s
val: New cache created: /home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/labels/val.cache
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.0001), 112 bias(decay=0.0)
Plotting labels to /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/yolov11m/labels.jpg... 
Image sizes 640 train, 640 val
Using 12 dataloader workers
Logging results to /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/yolov11m
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/150      7.86G      1.165      1.514      1.025         69        640: 100% ━━━━━━━━━━━━ 45/45 4.2it